<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab12_tool_grounded_single_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install Dependencies

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # All tools use Python stdlib only (re, json, datetime)
    print("Colab: no additional packages required.")
else:
    print("Local: stdlib-only — no installation needed.")

Colab: no additional packages required.


## 2. Data / Test Cases

Clone repo in Colab (`src/` files already in repo), add `src/` to sys.path.

In [2]:
import os, sys, warnings, json
from pathlib import Path

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("/content/NLP-Lab-works").exists():
        os.system("git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git /content/NLP-Lab-works")
    ROOT = Path("/content/NLP-Lab-works")
else:
    p = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (p / "src" / "tools.py").exists():
            ROOT = p
            break
        p = p.parent
    if ROOT is None:
        raise FileNotFoundError(f"Cannot locate repo root from {Path.cwd()}")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"ROOT: .../{ROOT.name}")
print("Source modules ready.")

ROOT: .../NLP-Lab-works
Source modules ready.


In [3]:
from eval_agent import TEST_CASES, BASELINE_RESPONSES, EXPECTED_OUTCOMES

print(f"Test cases: {len(TEST_CASES)}")
print()
print(f"{'#':<5} {'task_id':<12} {'scenario':<25} note")
print("-" * 90)
for i, tc in enumerate(TEST_CASES, 1):
    print(f"{i:<5} {tc['task_id']:<12} {tc['scenario']:<25} {tc['note'][:45]}")

Test cases: 10

#     task_id      scenario                  note
------------------------------------------------------------------------------------------
1     case_001     simple                    Clear entities + clear category — tools obvio
2     case_002     missing_data              No named entities — tools correctly return em
3     case_003     noisy_text                Misspellings — keyword tools miss entities; b
4     case_004     empty_result              No known entities at all — extract returns em
5     case_005     unnecessary_tool          Simple 1-entity text — validate called but ad
6     case_006     ambiguous                 Equal christian + electronics keyword hits — 
7     case_007     two_tools_sequential      Rich entities — extract then validate both ne
8     case_008     validator_finds_problem   4 christian + 4 electronics keywords → ambigu
9     case_009     answer_relies_on_tool     Final answer directly cites tool extraction f
10    case_010     tool_

## 3. Tool Definitions

Three tools implemented in `src/tools.py`:
- **`extract_entities(text)`** — regex + keyword NER (PERSON, ORG, GPE, DATE)
- **`classify_category(text)`** — keyword-score classifier for 3 newsgroup categories
- **`validate_extraction(data)`** — schema + consistency validator

Each tool: clear name → typed input → structured dict output → error handling.

In [4]:
from tools import extract_entities, classify_category, validate_extraction

# ── Demo: extract_entities ────────────────────────────────────────────────────
demo_text = (
    "Pope John Paul II visited Poland in June 1979. "
    "The Vatican is the seat of the Catholic Church in Italy."
)
ents = extract_entities(demo_text)
print("extract_entities demo:")
print(f"  persons       : {ents['persons']}")
print(f"  organizations : {ents['organizations']}")
print(f"  locations     : {ents['locations']}")
print(f"  dates         : {ents['dates']}")
print(f"  raw_count     : {ents['raw_count']}")

extract_entities demo:
  persons       : ['Pope John Paul II']
  organizations : ['Catholic Church', 'Vatican']
  locations     : ['Poland', 'Italy']
  dates         : ['June 1979']
  raw_count     : 6


In [5]:
# ── Demo: classify_category ───────────────────────────────────────────────────
clf = classify_category(demo_text)
print("classify_category demo:")
print(f"  category     : {clf['category']}")
print(f"  confidence   : {clf['confidence']}")
print(f"  scores       : {clf['scores']}")
print(f"  is_ambiguous : {clf['is_ambiguous']}")

print()
# Ambiguous case
ambig_text = (
    "Jesus Christ is the focus of faith. "
    "The circuit board and transistor are the main electronics components."
)
clf2 = classify_category(ambig_text)
print("Ambiguous case:")
print(f"  category   : {clf2['category']}")
print(f"  scores     : {clf2['scores']}")
print(f"  is_ambiguous: {clf2['is_ambiguous']}")

classify_category demo:
  category     : soc.religion.christian
  confidence   : 1.0
  scores       : {'sci.electronics': 0, 'soc.religion.christian': 3, 'alt.atheism': 0}
  is_ambiguous : False

Ambiguous case:
  category   : ambiguous
  scores     : {'sci.electronics': 3, 'soc.religion.christian': 3, 'alt.atheism': 0}
  is_ambiguous: True


In [6]:
# ── Demo: validate_extraction ────────────────────────────────────────────────
valid_data = {
    "category": "soc.religion.christian",
    "persons": ["Pope John Paul II"],
    "organizations": ["Vatican", "Catholic Church"],
    "locations": ["Poland", "Italy"],
    "dates": ["June 1979"],
}
v1 = validate_extraction(valid_data)
print("Valid case:")
print(f"  valid={v1['valid']}  errors={v1['errors']}  warnings={v1['warnings']}")

# Invalid: ambiguous category
invalid_data = {**valid_data, "category": "ambiguous"}
v2 = validate_extraction(invalid_data)
print()
print("Invalid case (ambiguous category):")
print(f"  valid={v2['valid']}  errors={v2['errors']}")

# Validator catches all-empty extraction
empty_data = {
    "category": "alt.atheism",
    "persons": [], "organizations": [], "locations": [], "dates": [],
}
v3 = validate_extraction(empty_data)
print()
print("Empty entities case:")
print(f"  valid={v3['valid']}  warnings={v3['warnings']}")

Valid case:
  valid=True  errors=[]  warnings=[]

Invalid case (ambiguous category):
  valid=False  errors=["Category is 'ambiguous' — cannot route to specific newsgroup"]

Empty entities case:
  valid=True  warnings=['All entity lists are empty — extraction may be incomplete']


## 4. Tool Call Logger

`ToolCallLogger` records every call with:
`timestamp`, `task_id`, `tool_name`, `input`, `output`, `success`, `error`, `reason`.

Two usage modes:
- **Manual**: `logger.log(...)` — after calling the tool yourself
- **Auto**: `logger.call(...)` — calls the tool and logs in one step (used by agent)

In [7]:
from tool_logger import ToolCallLogger

demo_logger = ToolCallLogger()

# Auto call + log
output, entry = demo_logger.call(
    task_id="demo",
    tool_name="extract_entities",
    tool_fn=extract_entities,
    input_data={"text": "Intel released its microprocessor in November 1971."},
    reason="Demo extraction",
)
print("Log entry fields:", list(entry.keys()))
print(f"  timestamp : {entry['timestamp']}")
print(f"  tool_name : {entry['tool_name']}")
print(f"  success   : {entry['success']}")
print(f"  output    : {entry['output']}")

# Error case
_, err_entry = demo_logger.call(
    task_id="demo", tool_name="extract_entities",
    tool_fn=extract_entities, input_data={"text": ""},
    reason="Error demo",
)
print()
print(f"Error case: success={err_entry['success']}  error={err_entry['error']!r}")
print(repr(demo_logger))

Log entry fields: ['timestamp', 'task_id', 'tool_name', 'input', 'output', 'success', 'error', 'reason']
  timestamp : 2026-05-29T21:11:16
  tool_name : extract_entities
  success   : True
  output    : {'persons': [], 'organizations': ['Intel'], 'locations': [], 'dates': ['November 1971'], 'raw_count': 2}

Error case: success=False  error='Input text cannot be empty'
ToolCallLogger(calls=2, success_rate=50.0%)


## 5. Agent Design

`SingleAgent` in `src/agent.py`:

```
User input (text + task_id)
       ↓
Tool 1: extract_entities(text)   → persons, orgs, locations, dates
       ↓
Tool 2: classify_category(text)  → category, confidence, is_ambiguous
       ↓  [adaptive: skip if text too short and 0 entities]
Tool 3: validate_extraction(data) → valid, errors, warnings
       ↓
Synthesise final answer from tool outputs
       ↓
Tool call log (JSONL)
```

**Tool-selection policy**:
- Always: `extract_entities` → `classify_category`
- `validate_extraction` only if `raw_count ≥ 1` OR `len(text) > 60`
- Early abort if step 1 raises an exception

In [8]:
from agent import SingleAgent, AgentResult
from tool_logger import ToolCallLogger

# Create logger + agent
logger = ToolCallLogger()
agent  = SingleAgent(logger=logger)

# Quick smoke test on one case
test_text = (
    "Intel released its first microprocessor in November 1971. "
    "The MIT Media Lab has been doing great work on signal processing."
)
result = agent.run(test_text, task_id="smoke_test")
print("Smoke test result:")
print(f"  category     : {result.final_answer['category']}")
print(f"  confidence   : {result.final_answer['confidence']}")
print(f"  persons      : {result.final_answer['persons']}")
print(f"  organizations: {result.final_answer['organizations']}")
print(f"  dates        : {result.final_answer['dates']}")
print(f"  tools called : {result.tools_called}")
print(f"  status       : {result.final_answer['status']}")
print(f"  validation   : {result.final_answer['validation']}")

Smoke test result:
  category     : sci.electronics
  confidence   : 1.0
  persons      : []
  organizations: ['Intel', 'MIT Media Lab']
  dates        : ['November 1971']
  tools called : ['extract_entities', 'classify_category', 'validate_extraction']
  status       : ok
  validation   : valid


## 6. Baseline LLM Without Tools

Pre-computed responses simulating what a basic LLM would return with no tool access.
The baseline sometimes hallucinates entity names or misclassifies ambiguous texts.

In [9]:
from eval_agent import BASELINE_RESPONSES, TEST_CASES

print("Baseline LLM responses (no tools):")
print("=" * 80)
for tc in TEST_CASES:
    tid = tc["task_id"]
    bl  = BASELINE_RESPONSES[tid]
    ents = (
        len(bl.get("persons", []))
        + len(bl.get("organizations", []))
        + len(bl.get("locations", []))
        + len(bl.get("dates", []))
    )
    flag = " *** HALLUCINATION/ERROR ***" if "HALLUCINATION" in bl["note"] or "WRONG" in bl["note"] or "MISSED" in bl["note"] else ""
    print(f"[{tid}] category={bl['category']:<28} entities={ents}{flag}")
    print(f"       note: {bl['note'][:75]}")
    print()

Baseline LLM responses (no tools):
[case_001] category=sci.electronics              entities=3
       note: Baseline incomplete: drops 'November' from date

[case_002] category=sci.electronics              entities=0
       note: Baseline correct — trivial case, no entities

[case_003] category=alt.atheism                  entities=4 *** HALLUCINATION/ERROR ***
       note: HALLUCINATION: baseline corrects misspellings and fabricates 'Scotland' + '

[case_004] category=alt.atheism                  entities=0
       note: Baseline correct — matches agent

[case_005] category=sci.electronics              entities=1
       note: Baseline correct — trivial case

[case_006] category=soc.religion.christian       entities=1 *** HALLUCINATION/ERROR ***
       note: WRONG CATEGORY: baseline ignores electronics keywords, picks only religious

[case_007] category=soc.religion.christian       entities=5
       note: Baseline mostly correct but missed Vatican as org

[case_008] category=sci.electro

## 7. Agent With Tools

Run `SingleAgent` on all 10 test cases.
The agent calls tools, logs every call, and synthesises a final structured answer.

In [10]:
from agent import SingleAgent
from tool_logger import ToolCallLogger
from eval_agent import run_evaluation, TEST_CASES

# Fresh logger for the main evaluation run
main_logger = ToolCallLogger()
main_agent  = SingleAgent(logger=main_logger)

agent_results = run_evaluation(main_agent, TEST_CASES)

print("Agent results (with tools):")
print("=" * 80)
for r in agent_results:
    print(r.summary_line())

Agent results (with tools):
[case_001] OK   | tools=3 | category=sci.electronics              | entities=3
[case_002] OK   | tools=3 | category=sci.electronics              | entities=0
[case_003] OK   | tools=3 | category=alt.atheism                  | entities=0
[case_004] OK   | tools=3 | category=alt.atheism                  | entities=0
[case_005] OK   | tools=3 | category=sci.electronics              | entities=1
[case_006] OK   | tools=3 | category=ambiguous                    | entities=1
[case_007] OK   | tools=3 | category=soc.religion.christian       | entities=6
[case_008] OK   | tools=3 | category=ambiguous                    | entities=1
[case_009] OK   | tools=3 | category=sci.electronics              | entities=2
[case_010] FAIL | tools=1 | category=?                            | entities=0


## 8. Run 10 Test Cases — Side-by-side Comparison

Baseline (LLM only) vs Agent (LLM + tools).

In [11]:
from eval_agent import BASELINE_RESPONSES, EXPECTED_OUTCOMES

print(f"{'task_id':<12} {'expected':<28} {'baseline':<28} {'agent':<28} {'tools_helped'}")
print("-" * 108)
for r, tc in zip(agent_results, TEST_CASES):
    tid  = tc["task_id"]
    exp  = EXPECTED_OUTCOMES[tid]
    bl   = BASELINE_RESPONSES[tid]["category"]
    ag   = r.final_answer.get("category", "error")
    helped = str(exp["tools_helped"])
    print(f"{tid:<12} {exp['expected_cat']:<28} {bl:<28} {ag:<28} {helped}")

task_id      expected                     baseline                     agent                        tools_helped
------------------------------------------------------------------------------------------------------------
case_001     sci.electronics              sci.electronics              sci.electronics              True
case_002     sci.electronics              sci.electronics              sci.electronics              False
case_003     alt.atheism                  alt.atheism                  alt.atheism                  partial
case_004     alt.atheism                  alt.atheism                  alt.atheism                  False
case_005     sci.electronics              sci.electronics              sci.electronics              False
case_006     ambiguous                    soc.religion.christian       ambiguous                    True
case_007     soc.religion.christian       soc.religion.christian       soc.religion.christian       True
case_008     ambiguous               

In [12]:
# Detailed view for cases where tools clearly helped
interesting = ["case_006", "case_007", "case_008"]
print("Detailed comparison — cases where tools improved over baseline:")
print()
for r in agent_results:
    if r.task_id not in interesting:
        continue
    tc = next(t for t in TEST_CASES if t["task_id"] == r.task_id)
    bl = BASELINE_RESPONSES[r.task_id]
    print(f"[{r.task_id}] Scenario: {tc['scenario']}")
    print(f"  Input    : {tc['text'][:80]}...")
    print(f"  Baseline : category={bl['category']}, orgs={bl.get('organizations')}")
    print(f"  Agent    : category={r.final_answer['category']}, "
          f"orgs={r.final_answer['organizations']}, "
          f"val_errors={r.final_answer.get('val_errors')}")
    print(f"  Note     : {bl['note'][:75]}")
    print()

Detailed comparison — cases where tools improved over baseline:

[case_006] Scenario: ambiguous
  Input    : Jesus Christ is the focus of faith. The circuit board and transistor are the mai...
  Baseline : category=soc.religion.christian, orgs=[]
  Agent    : category=ambiguous, orgs=[], val_errors=["Category is 'ambiguous' — cannot route to specific newsgroup"]
  Note     : WRONG CATEGORY: baseline ignores electronics keywords, picks only religious

[case_007] Scenario: two_tools_sequential
  Input    : Pope John Paul II visited Poland in June 1979. The Vatican is the seat of the Ca...
  Baseline : category=soc.religion.christian, orgs=['Catholic Church']
  Agent    : category=soc.religion.christian, orgs=['Catholic Church', 'Vatican'], val_errors=[]
  Note     : Baseline mostly correct but missed Vatican as org

[case_008] Scenario: validator_finds_problem
  Input    : Jesus Christ taught about faith. The resistor and capacitor in the circuit opera...
  Baseline : category=sci.electr

## 9. Tool Call Logs

Print all logged calls and save to `docs/tool_logs_lab12.jsonl`.

In [13]:
logs = main_logger.get_logs()
print(f"Total tool call log entries: {len(logs)}")
print()

# Print full log for first 3 cases
for entry in logs:
    if entry["task_id"] not in ("case_001", "case_002", "case_010"):
        continue
    print(f"[{entry['task_id']}] {entry['tool_name']:<22}"
          f"success={entry['success']}  "
          f"error={entry['error']!r}")
    if entry["output"]:
        # Show compact output
        out_preview = {k: v for k, v in entry["output"].items()
                       if k not in ("probabilities",)}
        print(f"         output: {out_preview}")
    print()

Total tool call log entries: 28

[case_001] extract_entities      success=True  error=None
         output: {'persons': [], 'organizations': ['Intel', 'MIT Media Lab'], 'locations': [], 'dates': ['November 1971'], 'raw_count': 3}

[case_001] classify_category     success=True  error=None
         output: {'category': 'sci.electronics', 'confidence': 1.0, 'scores': {'sci.electronics': 3, 'soc.religion.christian': 0, 'alt.atheism': 0}, 'is_ambiguous': False}

[case_001] validate_extraction   success=True  error=None
         output: {'valid': True, 'errors': [], 'warnings': [], 'error_count': 0, 'warning_count': 0}

[case_002] extract_entities      success=True  error=None
         output: {'persons': [], 'organizations': [], 'locations': [], 'dates': [], 'raw_count': 0}

[case_002] classify_category     success=True  error=None
         output: {'category': 'sci.electronics', 'confidence': 1.0, 'scores': {'sci.electronics': 3, 'soc.religion.christian': 0, 'alt.atheism': 0}, 'is_ambiguou

In [14]:
# Save JSONL log file
log_path = ROOT / "docs" / "tool_logs_lab12.jsonl"
n_lines  = main_logger.save_jsonl(log_path)
print(f"Saved {n_lines} log entries to docs/tool_logs_lab12.jsonl")

# Verify: reload and count
lines = log_path.read_text(encoding="utf-8").strip().splitlines()
print(f"Verified: {len(lines)} lines in JSONL file")
# Show one representative line
sample = json.loads(lines[0])
print(f"Sample line keys: {list(sample.keys())}")

Saved 28 log entries to docs/tool_logs_lab12.jsonl
Verified: 28 lines in JSONL file
Sample line keys: ['timestamp', 'task_id', 'tool_name', 'input', 'output', 'success', 'error', 'reason']


## 10. Metrics

Required metrics per ЛР12:
1. Tool call success rate
2. Average tool calls per task
3. Tasks with useful tool use
4. Unnecessary tool call count
5. Final answer correctness

In [15]:
from eval_agent import compute_metrics

metrics = compute_metrics(agent_results, main_logger)

sep = "-" * 50
print("=" * 50)
print("  Tool-grounded Agent Metrics")
print("=" * 50)
print(f"  Total test cases          : {metrics['total_cases']}")
print(sep)
print(f"  Total tool calls          : {metrics['total_tool_calls']}")
print(f"  Successful calls          : {metrics['successful_calls']}")
print(f"  Failed calls              : {metrics['failed_calls']}")
print(f"  Tool call success rate    : {metrics['tool_call_success_rate']:.1%}  "
      f"({metrics['successful_calls']}/{metrics['total_tool_calls']})")
print(sep)
print(f"  Avg tool calls / task     : {metrics['avg_calls_per_task']:.1f}")
print(f"  Tool call counts by tool  :")
for tool, cnt in metrics["tool_counts"].items():
    print(f"    {tool:<28}: {cnt}")
print(sep)
print(f"  Tasks where tools helped  : {metrics['tasks_with_useful_tools']}/10  "
      f"= {metrics['tasks_with_useful_pct']}%")
print(f"  Unnecessary tool calls    : {metrics['unnecessary_tool_calls']}")
print(sep)
print(f"  Final correct             : {metrics['final_correct']}/10  "
      f"= {metrics['final_correct_pct']}%")
print(f"  Final partly correct      : {metrics['final_partial']}/10")
print(f"  Final wrong / error       : {metrics['final_wrong']}/10")
print("=" * 50)

  Tool-grounded Agent Metrics
  Total test cases          : 10
--------------------------------------------------
  Total tool calls          : 28
  Successful calls          : 27
  Failed calls              : 1
  Tool call success rate    : 96.4%  (27/28)
--------------------------------------------------
  Avg tool calls / task     : 2.8
  Tool call counts by tool  :
    extract_entities            : 10
    classify_category           : 9
    validate_extraction         : 9
--------------------------------------------------
  Tasks where tools helped  : 5/10  = 50.0%
  Unnecessary tool calls    : 1
--------------------------------------------------
  Final correct             : 8/10  = 80.0%
  Final partly correct      : 1/10
  Final wrong / error       : 1/10


In [16]:
# Baseline vs agent comparison summary
bl_correct = sum(
    1 for tid, out in EXPECTED_OUTCOMES.items()
    if out["baseline_correct"] is True
)
ag_correct = metrics["final_correct"]

print("Comparison: Baseline vs Agent")
print(f"  Baseline correct : {bl_correct}/10 = {bl_correct*10:.0f}%")
print(f"  Agent correct    : {ag_correct}/10 = {ag_correct*10:.0f}%")
print(f"  Improvement      : +{ag_correct - bl_correct} cases")
print()
print("Cases where agent improved over baseline:")
for tid, out in EXPECTED_OUTCOMES.items():
    if out["tools_helped"] is True:
        bl = BASELINE_RESPONSES[tid]
        print(f"  [{tid}] baseline={bl['category'][:20]:<22}"
              f"→ agent correctly: {out['expected_cat']}")

Comparison: Baseline vs Agent
  Baseline correct : 4/10 = 40%
  Agent correct    : 8/10 = 80%
  Improvement      : +4 cases

Cases where agent improved over baseline:
  [case_001] baseline=sci.electronics       → agent correctly: sci.electronics
  [case_006] baseline=soc.religion.christi  → agent correctly: ambiguous
  [case_007] baseline=soc.religion.christi  → agent correctly: soc.religion.christian
  [case_008] baseline=sci.electronics       → agent correctly: ambiguous
  [case_009] baseline=sci.electronics       → agent correctly: sci.electronics


## 11. Error Analysis

Structured analysis of all 10 test cases.

Error categories used:
- `none` — tool worked correctly
- `unnecessary_tool_call` — tool called but produced no useful output
- `tool_input_malformed` — input prevented tool from working correctly
- `tool_output_ignored` — tool output not used in final answer
- `tool_fails` — tool raised an exception
- `agent_hallucinates_beyond_tool` — final answer contradicts or extends tool output

In [17]:
ERROR_ANALYSIS = [
    {
        "task_id": "case_001", "scenario": "simple",
        "error_category": "none",
        "input_preview": "Intel released its first microprocessor in November 1971...",
        "expected_behavior": "Extract Intel, MIT Media Lab, November 1971; classify sci.electronics",
        "actual_tool_calls": "extract(✓) → classify(✓) → validate(✓)",
        "final_answer_status": "correct",
        "notes": "Tools worked perfectly. Structured output traceable to exact tool calls.",
        "possible_fix": "N/A",
    },
    {
        "task_id": "case_002", "scenario": "missing_data",
        "error_category": "unnecessary_tool_call",
        "input_preview": "The circuit board has a few resistors and capacitors...",
        "expected_behavior": "Classify sci.electronics; entity extraction returns empty",
        "actual_tool_calls": "extract(✓→empty) → classify(✓) → validate(✓→warning)",
        "final_answer_status": "correct",
        "notes": "Validate called on all-empty extraction — warning generated but no action taken. Tools add logging overhead without insight.",
        "possible_fix": "Skip validate when extraction is empty and text is routine",
    },
    {
        "task_id": "case_003", "scenario": "noisy_text",
        "error_category": "tool_input_malformed",
        "input_preview": "Richar Dawkin$$ wrote The God Delusin in 2oo6!!...",
        "expected_behavior": "Classify alt.atheism; extract 'Richard Dawkins', 'David Hume', '2006'",
        "actual_tool_calls": "extract(✓→empty) → classify(✓→alt.atheism/low conf) → validate(✓→warning)",
        "final_answer_status": "partly_correct",
        "notes": "Keyword matching requires exact substring — misspellings prevent entity extraction. Baseline 'fixes' typos but hallucinate 'Scotland' and '2006'.",
        "possible_fix": "Add fuzzy matching (difflib, rapidfuzz) for entity lookup",
    },
    {
        "task_id": "case_004", "scenario": "empty_result",
        "error_category": "none",
        "input_preview": "I think the argument for theism is fundamentally flawed...",
        "expected_behavior": "Classify alt.atheism; no entities",
        "actual_tool_calls": "extract(✓→empty) → classify(✓→alt.atheism) → validate(✓→warning)",
        "final_answer_status": "correct",
        "notes": "Tool correctly returned empty entity lists. Category classifier picked up 'theism' keyword.",
        "possible_fix": "N/A — correct behaviour",
    },
    {
        "task_id": "case_005", "scenario": "unnecessary_tool",
        "error_category": "unnecessary_tool_call",
        "input_preview": "The transistor was invented in 1947.",
        "expected_behavior": "Classify sci.electronics; extract date '1947'",
        "actual_tool_calls": "extract(✓→{dates:[1947]}) → classify(✓) → validate(✓→no issues)",
        "final_answer_status": "correct",
        "notes": "validate_extraction called because raw_count=1, but it found no errors or warnings. One unnecessary call.",
        "possible_fix": "Raise validate threshold: call only if raw_count >= 2 or text > 100 chars",
    },
    {
        "task_id": "case_006", "scenario": "ambiguous",
        "error_category": "none",
        "input_preview": "Jesus Christ is the focus of faith. The circuit board...",
        "expected_behavior": "Detect ambiguous category; extract Jesus Christ",
        "actual_tool_calls": "extract(✓) → classify(✓→ambiguous) → validate(✓→error)",
        "final_answer_status": "correct",
        "notes": "Agent correctly identified tied keyword scores. Validator flagged ambiguity as blocking error. Baseline incorrectly chose soc.religion.christian.",
        "possible_fix": "Add a disambiguation tool: score_confidence(extraction, classification)",
    },
    {
        "task_id": "case_007", "scenario": "two_tools_sequential",
        "error_category": "none",
        "input_preview": "Pope John Paul II visited Poland in June 1979...",
        "expected_behavior": "Extract 6 entities; classify soc.religion.christian",
        "actual_tool_calls": "extract(✓→6 entities) → classify(✓) → validate(✓→valid)",
        "final_answer_status": "correct",
        "notes": "Both tools needed sequentially — extract builds input for validate. Baseline missed Vatican as org.",
        "possible_fix": "N/A — best case",
    },
    {
        "task_id": "case_008", "scenario": "validator_finds_problem",
        "error_category": "none",
        "input_preview": "Jesus Christ taught about faith. The resistor and capacitor...",
        "expected_behavior": "Detect ambiguity; validate raises blocking error",
        "actual_tool_calls": "extract(✓→Jesus Christ) → classify(✓→ambiguous) → validate(✓→error)",
        "final_answer_status": "correct",
        "notes": "Validator correctly caught 'ambiguous' category as a blocking error. Baseline completely ignored the religious figure and mis-classified.",
        "possible_fix": "N/A — this is the key demonstration of validator utility",
    },
    {
        "task_id": "case_009", "scenario": "answer_relies_on_tool",
        "error_category": "none",
        "input_preview": "Hewlett-Packard makes excellent multimeters...",
        "expected_behavior": "Extract Hewlett-Packard + San Jose; classify sci.electronics",
        "actual_tool_calls": "extract(✓) → classify(✓) → validate(✓→valid)",
        "final_answer_status": "correct",
        "notes": "Final answer org+location fields directly cite tool output. Provenance is fully auditable via tool call log.",
        "possible_fix": "N/A",
    },
    {
        "task_id": "case_010", "scenario": "tool_fails",
        "error_category": "tool_fails",
        "input_preview": "(empty string)",
        "expected_behavior": "Agent gracefully handles tool failure; returns structured error",
        "actual_tool_calls": "extract(✗→ValueError) → [pipeline aborted]",
        "final_answer_status": "partial",
        "notes": "Tool correctly raised ValueError for empty input. Agent caught it and aborted with error. classify_category not called — saves 1 unnecessary call. Error is fully logged.",
        "possible_fix": "Add pre-validation guard in agent.run(): check for empty text before calling any tool",
    },
]

print("Error Analysis — all 10 cases")
print("=" * 90)
print(f"{'#':<3} {'task_id':<12} {'error_category':<28} {'status':<15} notes")
print("-" * 90)
for i, e in enumerate(ERROR_ANALYSIS, 1):
    print(f"{i:<3} {e['task_id']:<12} {e['error_category']:<28} {e['final_answer_status']:<15} {e['notes'][:40]}")

Error Analysis — all 10 cases
#   task_id      error_category               status          notes
------------------------------------------------------------------------------------------
1   case_001     none                         correct         Tools worked perfectly. Structured outpu
2   case_002     unnecessary_tool_call        correct         Validate called on all-empty extraction 
3   case_003     tool_input_malformed         partly_correct  Keyword matching requires exact substrin
4   case_004     none                         correct         Tool correctly returned empty entity lis
5   case_005     unnecessary_tool_call        correct         validate_extraction called because raw_c
6   case_006     none                         correct         Agent correctly identified tied keyword 
7   case_007     none                         correct         Both tools needed sequentially — extract
8   case_008     none                         correct         Validator correctly caught '

In [18]:
# Category summary
from collections import Counter

cat_counts = Counter(e["error_category"] for e in ERROR_ANALYSIS)
print("Error category distribution:")
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:<30}: {cnt}")

print()
print("Status distribution:")
stat_counts = Counter(e["final_answer_status"] for e in ERROR_ANALYSIS)
for stat, cnt in sorted(stat_counts.items(), key=lambda x: -x[1]):
    print(f"  {stat:<20}: {cnt}")

print()
print("Possible fixes summary:")
for e in ERROR_ANALYSIS:
    if e["possible_fix"] != "N/A":
        print(f"  [{e['task_id']}] {e['possible_fix']}")

Error category distribution:
  none                          : 6
  unnecessary_tool_call         : 2
  tool_input_malformed          : 1
  tool_fails                    : 1

Status distribution:
  correct             : 8
  partly_correct      : 1
  partial             : 1

Possible fixes summary:
  [case_002] Skip validate when extraction is empty and text is routine
  [case_003] Add fuzzy matching (difflib, rapidfuzz) for entity lookup
  [case_004] N/A — correct behaviour
  [case_005] Raise validate threshold: call only if raw_count >= 2 or text > 100 chars
  [case_006] Add a disambiguation tool: score_confidence(extraction, classification)
  [case_007] N/A — best case
  [case_008] N/A — this is the key demonstration of validator utility
  [case_010] Add pre-validation guard in agent.run(): check for empty text before calling any tool


## 12. Generate `docs/audit_summary_lab12.md`

In [19]:

audit = f"""# Audit Summary — Lab 12: Tool-grounded Single Agent

**Date:** 2026-05-29

## 1. Use Case
NLP Research Post Analyzer — structured extraction from 20 Newsgroups posts.
Builds on corpus from ЛР10 (NER) and ЛР11 (LLM extraction schema-first).

## 2. Tools Implemented
- extract_entities(text)     — regex + keyword NER (PERSON, ORG, GPE, DATE)
- classify_category(text)    — keyword-score newsgroup classifier (3 categories)
- validate_extraction(data)  — schema + consistency validator

## 3. Test Cases
10 cases covering: simple, missing_data, noisy_text, empty_result,
unnecessary_tool, ambiguous, two_tools_sequential, validator_finds_problem,
answer_relies_on_tool, tool_fails.

## 4. Tool Call Success Rate
27 / 28 = 96.4%
Failed: 1 (case_010 extract_entities raised ValueError for empty input)

## 5. Average Tool Calls per Task
2.8 (28 calls / 10 tasks)

## 6. Tasks That Benefited from Tools
5 / 10 = 50%
Best cases: ambiguity detection (006, 008), complete entity extraction (007),
no-hallucination extraction (001, 009).

## 7. Unnecessary Tool Calls
1 — case_005: validate_extraction called on trivial 1-date result with no issues.

## 8. Best Tool Use Examples
- case_007: agent found Vatican as org (missed by baseline) + 6 total entities
- case_006: agent correctly returned "ambiguous" vs baseline's wrong single category
- case_008: validator caught ambiguous category as blocking error; baseline missed entirely

## 9. Problematic Examples
- case_003: noisy text with misspellings — keyword tools returned empty; baseline hallucinated
- case_010: empty input — tool raised, agent aborted (graceful but unhelpful)
- case_005: unnecessary validate call; trivial text needed only 2 tools

## 10. What to Improve Next
1. Fuzzy entity matching (rapidfuzz) for noisy/misspelled text
2. Dynamic tool selection via LLM planner instead of rule-based policy
3. Cross-field consistency check in validator (entity types vs category)
4. Pre-flight guard: reject empty/null inputs before calling any tool
5. Disambiguation tool for ambiguous category cases
"""

audit_path = ROOT / "docs" / "audit_summary_lab12.md"
audit_path.write_text(audit, encoding="utf-8")
print(f"Saved: docs/audit_summary_lab12.md")

Saved: docs/audit_summary_lab12.md
